# APM_50441_EP - Machine Learning 2 - Lab 1: Risk estimation, weights, hyperparameters and time series

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/APM-50441-EP/APM_50441_EP-2026-2027/blob/main/PC1/APM_50441_EP_Lab_1_simplified.ipynb) or run it on your own computer (see *Working environment* below).

This lab illustrates the first lecture (*Setting, Risk Estimation... and Time Series*). Its goals are:
- to estimate the risk of a predictor and to measure the uncertainty of this estimation (hold-out, cross validation, comparison of two models, selection rules),
- to take into account a difference between the training and the real-world proportions, and a cost depending on the prediction (weights, Bayes rule),
- to compare hyperparameter search strategies with a fixed budget, and to see how much the selection itself can be overfitted,
- to validate a forecast correctly when the data are a time series.

We use `Python` with `scikit-learn`, `polars` and `seaborn`.

**Working environment.** Colab is the simplest option: open the notebook of this lab from the course repository (*Open in Colab* link) and run it. To work on your own computer, download the lab folder (the notebook and its files) or clone the course repository, and open the notebook in an IDE that runs Jupyter notebooks (VS Code with the Python and Jupyter extensions, JupyterLab, Spyder, PyCharm). Create a Python environment once with [`uv`](https://docs.astral.sh/uv/): `uv venv`, activate it (`source .venv/bin/activate`, or `.venv\Scripts\activate` on Windows) and `uv pip install jupyter`, or build the whole course environment with `uv sync` in `PC/env`; then select this environment as the kernel of the notebook. The installation cell below installs the packages of the lab into the Python running the notebook (Colab or your environment), and the data are read from the local files when they are present (otherwise they are downloaded).

**Use of AI.** AI assistants (e.g. the ones built into Colab or VS Code) are allowed, but you must be able to explain every line you use, and you should answer the *predict before you run* questions before running the code: the suggested code may not be correct!

All the libraries of the core part are installed at once. On Colab, restart the session once after this installation; locally, restart the kernel if the installation upgraded a package that was already loaded.

In [ ]:
# Installs into the Python running this notebook: Colab, or locally an activated virtual environment
import sys
!uv pip install --python "{sys.executable}" "scikit-learn==1.9.1" "polars==1.44.2" "seaborn==0.13.2" "fairlearn==0.14.0" "optuna==5.0.0" "statsmodels==0.15.0"

We use the Credit Risk dataset prepared by IBM (a biased version of the classical German credit data), and `apm_credit.py`, a small module gathering the cost model that we build step by step in this lab, so that the next labs can simply import it. Each file is downloaded only if there is no local copy.

In [ ]:
import os
import urllib.request

# each file is downloaded only if there is no local copy
CSV_URL = "https://raw.githubusercontent.com/IBM/monitor-wml-model-with-watson-openscale/master/data/german_credit_data_biased_training.csv"
APM_CREDIT_URL = "https://raw.githubusercontent.com/APM-50441-EP/APM_50441_EP-2026-2027/main/common/apm_credit.py"  # course repository
for url in [CSV_URL, APM_CREDIT_URL]:
  filename = url.split('/')[-1]
  if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)

The dataset, the training/test split and two pipelines (a logistic regression and a nearest neighbors classifier) are given: this was the content of the first labs of the previous course.

In [ ]:
import numpy as np
import polars as pl
import polars.selectors as cs
import seaborn.objects as so
import matplotlib.pyplot as plt

import sklearn
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from threadpoolctl import threadpool_limits
threadpool_limits(limits=2, user_api='openmp')  # on such small datasets, many threads are slower than a few (Colab has 2 cores)

german_credit = pl.read_csv("german_credit_data_biased_training.csv")
X = german_credit.drop('Risk')
y = german_credit['Risk']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

categorical_features = X.select(cs.string()).columns
numerical_features = X.select(cs.numeric()).columns
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ('num', StandardScaler(), numerical_features)])

def make_pipeline(classifier):
  return Pipeline(steps=[('preprocessor', clone(preprocessor)), ('classifier', classifier)]).set_output(transform="polars")

pipeline_log = make_pipeline(LogisticRegression(max_iter=1000))
pipeline_nn = make_pipeline(KNeighborsClassifier(n_neighbors=15))
german_credit.shape, y_train.value_counts(normalize=True).sort('Risk')

## 1. Risk estimation and model comparison

The *risk* of a predictor $f$ is its average loss on a new observation, $\mathcal{R}(f) = \mathbb{E}[\ell(Y, f(X))]$. It is unknown and must be estimated from data that have not been used to train $f$.

**Question** *Predict before you run.* For the $1$-nearest neighbor, the $15$-nearest neighbors and the logistic regression, what do you expect for the accuracy on the training set compared with the accuracy on the test set? Check it.

**Answer**

---

A single test set gives a single, noisy, estimate. To see how noisy, we can repeat the *hold-out* procedure: split the training set into a training part and a validation part many times.

**Question** Estimate the accuracy of the logistic regression with $20$ random hold-out splits ($80\%$/$20\%$) of the training set, and with a $5$-fold cross validation repeated $4$ times (`StratifiedKFold(n_splits=5, shuffle=True, random_state=i)`). Compare the spread of the individual estimates, and of the averages.

**Answer**

---

The standard deviation of the fold scores describes the variability of the accuracy of *one* fitted model on *one* validation set; the standard error (the standard deviation divided by $\sqrt{V}$) describes the variability of their *mean*, under an independence assumption that does not hold (the training parts of the folds overlap).

**Question** Compare the logistic regression and the $15$-nearest neighbors on the *same* folds ($5$-fold CV repeated $4$ times). Compute the paired differences, and compare a naive $t$-test, which assumes independent differences, with the *corrected resampled $t$-test* of Nadeau and Bengio, which inflates the variance of the mean difference from $s^2/J$ to $(1/J + n_{\text{val}}/n_{\text{train}})\, s^2$.

**Answer**

---

**Question** Choose the number of neighbors $k$ among the values below with a $5$-fold cross validation, with three rules: the best average accuracy, the simplest model (largest $k$) whose average is within one standard error of the best one (*one standard error rule*), and the best *worst fold* (a pessimistic, quantile-based rule in the spirit of the PAC approach of the lecture). Evaluate the selected models once on the test set.

In [ ]:
n_neighbors_values = [1, 3, 5, 9, 15, 25, 41, 67, 111, 181, 301]

**Answer**

---

## 2. Unbalanced data, weights and costs

In our dataset, about one third of the loans are `Risk`, whereas we assume that in the real world the proportion of `Risk` is $\pi_{\text{Risk}} = 0.02$. If the conditional law of $X$ given $Y$ is the same in both worlds, an expectation in the real world is a *weighted* expectation in the training world, with a weight $w_y = \pi_y / \pi^{\text{train}}_y$ depending only on the class:
$$\mathbb{E}_{\text{real}}[\ell(Y, f(X))] = \mathbb{E}_{\text{train}}[w_Y\, \ell(Y, f(X))].$$

In [ ]:
real_prop = {'Risk': .02, 'No Risk': .98}
train_prop = dict(y_train.value_counts(normalize=True).iter_rows())
custom_weight = {c: real_prop[c] / train_prop[c] for c in real_prop}
train_prop, custom_weight

**Question** Write a function `custom_metric(y_true, y_pred)` computing the real-world accuracy (the weighted accuracy) and the corresponding scorer. What is the real-world accuracy of the logistic regression? Of a classifier that always predicts `No Risk`?

**Answer**

---

Two ways of adapting the predictor are seen in the lecture: training with the weights (*weighted loss*), or keeping the probabilities estimated in the training world and changing the decision (*plug-in Bayes rule*): predict `Risk` when $w_{\text{Risk}}\, \hat{p}(\text{Risk}|x) > w_{\text{No Risk}}\, \hat{p}(\text{No Risk}|x)$.

To send weights to a step of a pipeline, `scikit-learn` uses *metadata routing*: each step declares whether it uses them.

**Question** Train the logistic regression with the weights (only the classifier should use them), and implement the plug-in decision. Compare their real-world accuracy on the test set.

**Answer**

---

The accuracy is not the right criterion here: the cost of an error depends on the error and on the amount of the loan. We use the following cost model (the opposite of a gain):
- $5 + 0.6 \times \text{LoanAmount}$ if we grant a loan that defaults (`Risk` predicted `No Risk`),
- $1 - 0.05 \times \text{LoanAmount}$ if we grant a good loan (`No Risk` predicted `No Risk`): a benefit of $0.05 \times \text{LoanAmount}$ after a fixed cost of $1$,
- $1$ if we refuse the loan.

The cost model, the weighted cost metric and the corresponding scorers are gathered in `apm_credit.py`. The scorers return the *gain* (the opposite of the cost), and need `LoanAmount` as metadata:

In [ ]:
import apm_credit
from apm_credit import cost_metric, cost_scorer, cost_scorer_from_proba, compute_cost_weights, compute_costs
apm_credit.set_proportions(y_train, real_prop)
LoanAmount_train, LoanAmount_test = X_train['LoanAmount'], X_test['LoanAmount']
cost_scorer(pipeline_log, X_test, y_test, LoanAmount=LoanAmount_test)

**Question** *Predict before you run.* Write the Bayes rule for this cost: for a loan of amount $a$ and an estimated probability $p$ of `Risk`, when should we refuse it? Compute the threshold on $p$ for a loan of $5000$, and compare it with the threshold $1/2$ of `predict`.

**Answer**

---

**Question** Compare, with a $5$-fold cross validation of the gain on the training set, the original logistic regression, the logistic regression trained with the cost weights `compute_cost_weights(y, LoanAmount)`, the Bayes decision on the probabilities of the original one (`cost_scorer_from_proba`), and a threshold tuned by cross validation (`TunedThresholdClassifierCV`). Which one would you choose? Evaluate it on the test set.

**Answer**

---

## 3. Hyperparameter optimization with a fixed budget

We now tune a `HistGradientBoostingClassifier` (a model of Lecture 3, fast to train) for the cost-based gain of the Bayes decision. The lecture presents several strategies: an exhaustive grid, random search, *successive halving* (many candidates evaluated on small budgets, the best ones promoted to larger budgets), and Bayesian optimization (here Optuna's Tree-structured Parzen Estimator, TPE). We are going to compare them with roughly the same budget of $20$ full $3$-fold cross validations.

Note: `HistGradientBoostingClassifier` already uses several cores (we limited them to $2$ at the beginning of the lab); running several cross validations in parallel (`n_jobs`) on top of it oversubscribes them and is often *slower*, so we keep `n_jobs=None` here.

In [ ]:
import time
import optuna
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.experimental import enable_halving_search_cv  # noqa: F401 (successive halving is still experimental)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, HalvingRandomSearchCV
from scipy.stats import loguniform, randint

pipeline_hgb = make_pipeline(HistGradientBoostingClassifier(random_state=0))
cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)
grid = {'classifier__learning_rate': [0.01, 0.05, 0.2, 1.0], 'classifier__max_leaf_nodes': [3, 7, 15, 31, 63]}
distributions = {'classifier__learning_rate': loguniform(0.01, 1), 'classifier__max_leaf_nodes': randint(3, 64)}

**Question** Run a grid search on the $4 \times 5$ grid `grid`, a random search with $20$ draws from `distributions`, a successive halving search (`HalvingRandomSearchCV` with $60$ candidates, `factor=3` and `min_resources='exhaust'`, the resource being the number of training samples) and an Optuna study with $20$ trials of a TPE sampler, all with the gain of the Bayes decision (`cost_scorer_from_proba`) as score. Compare their best CV gains, their running times, and the gains of the selected models on the test set.

**Answer**

---

The best CV score of a search is the maximum of noisy estimates: it is optimistic, and the more candidates are tried, the more optimistic it is (*overfitting the validation set*). To see this *selection bias* alone, we select the configuration with a first cross validation (folds A), and re-estimate the selected configuration with an independent second cross validation (folds B, another shuffle of the same data): the configuration does not change, only the noise does. As in the lecture, the phenomenon is much clearer with a small training set: we use $400$ loans of the training set.

**Question** Draw $60$ random configurations from `distributions`. For each of them, compute the $3$-fold CV gain on the small training set with two different shuffles (A and B). As a function of the number $m$ of configurations tried, plot the best CV gain on A (the reported one) and the CV gain on B of the configuration selected on A. Comment.

**Answer**

---

## 4. Validation of a time series forecast

We forecast the monthly CO$_2$ concentration at Mauna Loa (1958–2001, bundled with `statsmodels`) **one year ahead**: at the end of month $t$, we want $y_{t+12}$. Only the values up to time $t$ can be used, so the features of the target $y_s$ can only use $y_{s-12}, y_{s-13}, \dots$ (the *lags* larger than the horizon), the calendar (month) and the time.

The last $10$ years (1992–2001) are kept as a test period.

In [ ]:
import statsmodels.api as sm
co2 = sm.datasets.co2.load_pandas().data['co2'].resample('MS').mean().interpolate()
co2 = pl.DataFrame({'date': co2.index.to_numpy(), 'co2': co2.to_numpy()})
H = 12  # forecast horizon (months)
co2_features = (co2.with_columns(pl.col('date').dt.month().alias('month'),
                                 ((pl.col('date').dt.year() - 1958) + (pl.col('date').dt.month() - 1) / 12).alias('time'),
                                 *[pl.col('co2').shift(lag).alias(f'lag_{lag}') for lag in range(H, H + 13)])
                .with_columns(*[f(2 * np.pi * k * pl.col('month') / 12).alias(f'{f.__name__}_{k}')
                                for k in (1, 2) for f in (np.sin, np.cos)])
                .drop_nulls())
train_ts = co2_features.filter(pl.col('date').dt.year() < 1992)
test_ts = co2_features.filter(pl.col('date').dt.year() >= 1992)
features = ['time', 'sin_1', 'cos_1', 'sin_2', 'cos_2'] + [f'lag_{lag}' for lag in range(H, H + 13)]
so.Plot(co2, x='date', y='co2').add(so.Line())

**Question** The *seasonal naive* forecast predicts $y_{t+12}$ by $y_t$, i.e. $\hat{y}_s = y_{s-12}$ (`lag_12`). Compute its mean absolute error (MAE) on the test period. Then fit a ridge regression and a `HistGradientBoostingRegressor` on the `features`, and compute their test MAE.

**Answer**

---

**Question** *Predict before you run.* We want to choose between these models with the training period only. Compare the MAE estimated by a shuffled $5$-fold cross validation, by `TimeSeriesSplit(n_splits=5, test_size=24, gap=H)` (each validation block comes after its training part, with a gap of at least the horizon), and the test MAE. Which estimation is reliable?

**Answer**

---

**Question** Trees cannot extrapolate: the gradient boosting predicts values in the range of its training targets. *Stationarize* the series by forecasting the seasonal difference $y_{t+12} - y_t$ (i.e. predict `co2 - lag_12` and add `lag_12` back), and compare the test MAE of both models.

**Answer**

---

## About this lab

**Use of AI.** Artificial intelligence tools have been used to revise the content of this lab. The main contributor takes full responsibility for this content.

**License.** The texts of this lab are under the [Creative Commons Attribution-ShareAlike 4.0](https://creativecommons.org/licenses/by-sa/4.0/) license (CC BY-SA 4.0), and its code under the [MIT](https://opensource.org/license/mit) license; the datasets keep their own licenses. Main contributor: E. Le Pennec. Contributors: S. Boucheron, A. Dieuleveut, A.K. Fermin, S. Gadat, S. Gaiffas, B. Goujaud, A. Guilloux, Ch. Keribin, Th. Klein, E. Matzner, J. Read, M. Sangnier, E. Scornet.